<a href="https://colab.research.google.com/github/TheTrappist/teaching/blob/main/Biochem6761/AU26/05_AU26_EnzymeKinetics_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt
import warnings
from scipy.integrate import solve_ivp


The following numerical solver will integrate a system of differential equations that describes a simple enzyme-catalyzed reaction.

$$
\begin{aligned}
\text{Overall reaction scheme:}\quad
& \mathrm{E} + \mathrm{S}
\;\underset{k_{-1}}{\overset{k_1}{\rightleftharpoons}}\;
\mathrm{ES}
\;\underset{k_{-2}}{\overset{k_2}{\rightleftharpoons}}\;
\mathrm{EP}
\;\underset{k_{-3}}{\overset{k_3}{\rightleftharpoons}}\;
\mathrm{E} + \mathrm{P}
\end{aligned}
$$

$$
\text{Note: }{k_{-2}} \text{ and }{k_{-3}} \text{ are assumed to be zero (making these steps irreversible) unless specified otherwise.}
$$

$$
\begin{aligned}
\frac{d[\mathrm{E}]}{dt}  &= -k_1[\mathrm{E}][\mathrm{S}] + k_{-1}[\mathrm{ES}] + k_3[\mathrm{EP}] - k_{-3}[\mathrm{E}][\mathrm{P}] \\[6pt]
\frac{d[\mathrm{S}]}{dt}  &= -k_1[\mathrm{E}][\mathrm{S}] + k_{-1}[\mathrm{ES}] \\[6pt]
\frac{d[\mathrm{ES}]}{dt} &= k_1[\mathrm{E}][\mathrm{S}] - (k_{-1} + k_2)[\mathrm{ES}] + k_{-2}[\mathrm{EP}] \\[6pt]
\frac{d[\mathrm{EP}]}{dt} &= k_2[\mathrm{ES}] - (k_{-2} + k_3)[\mathrm{EP}] + k_{-3}[\mathrm{E}][\mathrm{P}] \\[6pt]
\frac{d[\mathrm{P}]}{dt}  &= k_3[\mathrm{EP}] - k_{-3}[\mathrm{E}][\mathrm{P}]
\end{aligned}
$$

$$
\text{Conservation laws:}\quad
[\mathrm{E}] + [\mathrm{ES}] + [\mathrm{EP}] = [\mathrm{E}]_{\mathrm{total}},
\qquad
[\mathrm{S}] + [\mathrm{ES}] + [\mathrm{EP}] + [\mathrm{P}] = [\mathrm{S}]_{\mathrm{total}}
$$



In [ ]:
# Set up ine the numerical integrator inside a wrapper function

def simulate_kinetics(k1, k_minus1, k2, k3, *,
                      E0, S0, ES0=0.0, EP0=0.0, P0=0.0,
                      duration,
                      k_minus2=0.0, k_minus3=0.0,
                      log_time=False, mm_overlay=False,
                      n_points=1000, conservation_tol=1e-6,
                      exclude_from_plot=()):
    """
    Simulate and plot the scheme:

        E + S <-> ES <-> EP <-> E + P
           k1/k-1    k2/k-2    k3/k-3

    Units: concentrations in µM, time in s.
        k1, k_minus3   : µM^-1 s^-1  (bimolecular)
        k_minus1, k2, k_minus2, k3 : s^-1  (unimolecular)

    Parameters
    ----------
    k1, k_minus1, k2, k3 : float
        Rate constants for E+S->ES, ES->E+S, ES->EP, EP->E+P.
    E0, S0, ES0, EP0, P0 : float
        Initial concentrations (µM).
    duration : float
        Length of the experiment (s).
    k_minus2, k_minus3 : float, optional
        Rate constants for EP->ES and E+P->EP (default 0 = irreversible).
    log_time : bool
        Use a logarithmic time axis (default False).
    mm_overlay : bool
        Overlay the quasi-steady-state Michaelis-Menten prediction for S and P
        (default False).
    n_points : int
        Number of time points for the output/plot.
    conservation_tol : float
        Relative tolerance for the mass-conservation check.
    exclude_from_plot : tuple of strings
        Names of species to exclude from final plot for clarity
    Returns
    -------
    t : ndarray
    conc : dict of ndarray, keys 'E', 'S', 'ES', 'EP', 'P'
    fig, ax : matplotlib figure and axes
    """

    # The ODE system is defined below (see markdown cell above for a
    # clean version of the equations used).
    def rhs(t, y):
        E, S, ES, EP, P = y
        v1 = k1 * E * S - k_minus1 * ES        # net E + S -> ES
        v2 = k2 * ES - k_minus2 * EP           # net ES -> EP
        v3 = k3 * EP - k_minus3 * E * P        # net EP -> E + P
        return [-v1 + v3,   # dE/dt
                -v1,        # dS/dt
                v1 - v2,    # dES/dt
                v2 - v3,    # dEP/dt
                v3]         # dP/dt

    y0 = [E0, S0, ES0, EP0, P0]

    # Build the time grid, either on a logarithmic scale or a linear one.
    if log_time:
        t = np.geomspace(duration * 1e-6, duration, n_points)
    else:
        t = np.linspace(0, duration, n_points)

    # Radau handles stiffness (fast binding vs. slow turnover) well
    sol = solve_ivp(rhs, (0, duration), y0, t_eval=t,
                    method="Radau", rtol=1e-9, atol=1e-12)
    if not sol.success:
        raise RuntimeError(f"Integration failed: {sol.message}")

    E, S, ES, EP, P = sol.y
    conc = {"E": E, "S": S, "ES": ES, "EP": EP, "P": P}

    # Calculate total amounts of enzyme and substrate+product

    E_tot0 = E0 + ES0 + EP0
    S_tot0 = S0 + ES0 + EP0 + P0

    # Generate the plot of the full reaction
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = {"E": "tab:blue", "S": "tab:orange", "ES": "tab:green",
              "EP": "tab:red", "P": "tab:purple"}
    labels = {"E": "E (free enzyme)", "S": "S (substrate)",
              "ES": "ES", "EP": "EP", "P": "P (product)"}
    for name, c in conc.items():
        if name not in exclude_from_plot:
            ax.plot(t, c, color=colors[name], lw=2, label=labels[name])

    #If requested, calculate and add a Michaelis-Menten overlay
    if mm_overlay:
        if k_minus2 != 0 or k_minus3 != 0:
            warnings.warn("MM overlay assumes irreversible ES->EP and "
                          "EP->E+P; it ignores k_minus2 and k_minus3.")
        # Steady-state treatment of ES and EP (irreversible steps):
        kcat = k2 * k3 / (k2 + k3)
        Km = k3 * (k_minus1 + k2) / (k1 * (k2 + k3))

        def mm_rhs(t_, y):
            S_, P_ = y
            v = kcat * E_tot0 * S_ / (Km + S_)
            return [-v, v]

        mm = solve_ivp(mm_rhs, (0, duration), [S_tot0 - P0, P0], t_eval=t,
                       rtol=1e-9, atol=1e-12)
        ax.plot(t, mm.y[0], "--", color=colors["S"], lw=1.5,
                label=f"S (MM, Km={Km:.3g} µM, kcat={kcat:.3g} s⁻¹)")
        ax.plot(t, mm.y[1], "--", color=colors["P"], lw=1.5, label="P (MM)")

    if log_time:
        ax.set_xscale("log")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Concentration (µM)")
    ax.set_title("Enzyme kinetics: E + S ⇌ ES ⇌ EP ⇌ E + P")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    return t, conc, fig, ax

In [ ]:
# Now, let's make some plots! The settings below produce a nice and manageable one
results = simulate_kinetics(
        k1=1.0,          # µM^-1 s^-1
        k_minus1=2.0,    # s^-1
        k2=5.0,          # s^-1
        k3=1.0,          # s^-1
        E0=5, S0=20.0,  # µM
        duration=10.0,    # s
        log_time=False,
        mm_overlay=True
    )

# Just to see where the data 'live': print the first ten data points of enzyme concentrations
print(results[1]['E'][0:10])

In [ ]:
# Play time!
results = simulate_kinetics(
        k1=0.1,          # µM^-1 s^-1
        k_minus1=2.0,    # s^-1
        k2=1.0,          # s^-1
        k3=1.0,          # s^-1
        k_minus2=0.0, 
        k_minus3=0.0,
        E0=20.0, S0=1.0,  # µM
        duration=10.0,    # s
        log_time=False,
        mm_overlay=False,
        exclude_from_plot = ('E') # Do not plot this
    )
